# 13 -- Cuantiles, calibración y sensibilidad de costos por estrato (fase E, parte 2)

Hasta aquí se evaluó el **q50** (WAPE). Lo que el negocio usa es el **cuantil de servicio** `alpha = Cu/(Cu+Co)` (0.893 para no perecederos con
Cu=0.25, Co=0.03; 0.167 para perecederos con Cu=0.20, Co=1.00; ambos costos validados por el negocio). Aquí se evalúan **cuantiles en una rejilla de
alphas**, por estrato de volumen, y se contrasta con un cuantil base **sin modelo** construido sobre la media móvil. Como ejercicio de modelado, los
costos se pueden variar: con `Cu=alpha`, `Co=1-alpha` el costo esperado del vendedor de periódicos coincide con la pérdida pinball, así que la
sensibilidad a `Cu/Co` es la sensibilidad al alpha.

- **Modelo** (`q_modelo`): LightGBM cuantílico sobre el target relativo `y/base` (`base = nivel_medio_60d·H+1`, el mejor global de la fase D), 14 features v1,
  hiperparámetros de cada rama, sin búsqueda, entrenamiento con purge.
- **Base por rama** (`qb`): `(media_movil + 1) · Q_alpha(y/(media_movil+1))`, con `Q_alpha` sobre las filas de entrenamiento (con purge) de la misma rama.
- **Base por estrato** (`qe`, la exigente): igual, pero con `Q_alpha` calculado **dentro de cada estrato as-of** (rama x estrato). La base por rama ignora que la cabeza tiene
  otra distribución de razones y es floja a alphas bajos; las decisiones se toman contra `qe`.
- **Producción** (`pred_q_negocio` de 08): en las ramas suaves es ensamble Tweedie+LightGBM con recalibración conformal, no el LightGBM relativo de este notebook; se compara aparte
  en la sección 4.
- Estratos as-of de `12_segmentacion_volumen`. Métrica: pérdida pinball agregada (la mejora se reporta relativa a la base, con IC95 por bootstrap de
  productos) y **cobertura empírica** (`P(y <= q)`), que debe acercarse a `alpha`.

Cadena: ... -> `12_segmentacion_volumen` -> **`13_cuantiles_y_costos`**

In [1]:
import sys
sys.path.insert(0, '.')
from common_priorizacion import DW, registrar_huella, verificar_huella, purgar_entrenamiento
import evaluar_nivel1 as ev
from exportar_modelos_nivel1 import FEATURES as F1, MODELOS_POR_RAMA, RAMA_A_CONDICION, COSTOS_SUPUESTOS

import json, time, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
warnings.filterwarnings('ignore')

pd.set_option('display.width', 230)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

EXPERIMENTO = 'EDA-fix-nivel1-13-cuantiles-y-costos'
Y, PISO, H = 'target_demanda_15d', 'pred_media_movil_15d', 15
LLAVES = ['codigo_item', 'sucursal', 'fecha_origen']
ALPHAS = [0.1, 0.167, 0.25, 0.5, 0.75, 0.893, 0.95]
ESTRATOS = ['cabeza', 'medio', 'cola']
B, SEMILLA = 2000, 0
ALPHA_RAMA = {'intermitente': 0.893, 'suave_no_perecedero': 0.893, 'suave_perecedero': 0.167}   # como en 08 / exportar_modelos_nivel1
COSTO_RAMA = {'intermitente': COSTOS_SUPUESTOS['no_perecedero'], 'suave_no_perecedero': COSTOS_SUPUESTOS['no_perecedero'],
              'suave_perecedero': COSTOS_SUPUESTOS['perecedero']}

verificar_huella('huella_07.json', ['matriz_as_of.parquet'])
verificar_huella('huella_12.json', ['estratos_por_fold_12.parquet'])
verificar_huella('huella_08.json', ['evaluacion_nivel_1.parquet'])
m = pd.read_parquet(f'{DW}/matriz_as_of.parquet')
estr = pd.read_parquet(f'{DW}/estratos_por_fold_12.parquet')
estr['codigo_item'], estr['sucursal'] = estr['codigo_item'].astype(str), estr['sucursal'].astype(str)

def etiqueta_estrato(df, f):
    e = estr[estr['fold_id'] == f].set_index(['codigo_item', 'sucursal'])['estrato']
    return e.reindex(pd.MultiIndex.from_arrays([df['codigo_item'].astype(str), df['sucursal'].astype(str)])).to_numpy()
print(m.shape, estr.shape)

/home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/24 19:46:48 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Huella verificada OK contra huella_07.json.
Huella verificada OK contra huella_12.json.
Huella verificada OK contra huella_08.json.
(262429, 27) (44540, 5)


## 1. Entrenamiento walk-forward por rama, fold y alpha

In [2]:
def hp_lgb(rama):
    s = MODELOS_POR_RAMA[rama]
    return s['hparams'] if s['tipo'] == 'lightgbm' else s['hparams_lgb']

partes, t0 = [], time.time()
for rama, cond in RAMA_A_CONDICION.items():
    dr = m.loc[cond(m)]
    for f in [2, 3, 4, 5]:
        tr, te = purgar_entrenamiento(dr, f, verbose=False), dr[dr['fold_id'] == f]
        if len(tr) < ev.MIN_FILAS_ENTRENAMIENTO:
            print(f'{rama:22s} fold {f}: train={len(tr)} < {ev.MIN_FILAS_ENTRENAMIENTO} -- omitida (igual que 08)'); continue
        base_tr, base_te = tr['nivel_medio_60d'] * H + 1.0, te['nivel_medio_60d'] * H + 1.0
        ratio_tr = tr[Y] / (tr[PISO] + 1.0)
        e_tr, e_te = etiqueta_estrato(tr, f), etiqueta_estrato(te, f)
        blk = te[LLAVES + ['fold_id']].copy()
        blk['y'], blk['piso'], blk['rama'], blk['estrato'] = te[Y].to_numpy(), te[PISO].to_numpy(), rama, e_te
        for a in ALPHAS:
            mod = lgb.LGBMRegressor(objective='quantile', alpha=a, subsample_freq=1, random_state=42, verbose=-1, **hp_lgb(rama))
            mod.fit(tr[F1], tr[Y] / base_tr)
            blk[f'qm_{a}'] = np.clip(mod.predict(te[F1]) * base_te.to_numpy(), 0, None)
            blk[f'qb_{a}'] = float(np.quantile(ratio_tr, a)) * (te[PISO].to_numpy() + 1.0)
            qmap = {e: (float(np.quantile(ratio_tr[e_tr == e], a)) if (e_tr == e).sum() >= 30 else float(np.quantile(ratio_tr, a))) for e in ESTRATOS}
            blk[f'qe_{a}'] = pd.Series(e_te).map(qmap).to_numpy() * (te[PISO].to_numpy() + 1.0)
        partes.append(blk)
        print(f'{rama:22s} fold {f}: train={len(tr):>7,} test={len(te):>6,} ({time.time()-t0:,.0f}s)')
res = pd.concat(partes, ignore_index=True)
for c in ('codigo_item', 'sucursal'):
    res[c] = res[c].astype(str)
assert res['estrato'].notna().all(), 'hay pares de prueba sin estrato'
print(res.shape)

suave_no_perecedero    fold 2: train=    260 test=   871 (5s)


suave_no_perecedero    fold 3: train=  1,129 test=   315 (11s)


suave_no_perecedero    fold 4: train=  1,448 test=   600 (18s)


suave_no_perecedero    fold 5: train=  2,051 test=   624 (25s)
suave_perecedero       fold 2: train=168 < 200 -- omitida (igual que 08)


suave_perecedero       fold 3: train=    714 test=   200 (28s)


...

suave_perecedero       fold 5: train=  1,316 test=   444 (35s)


intermitente           fold 2: train= 20,152 test=68,198 (53s)


intermitente           fold 3: train= 88,142 test=28,850 (81s)


intermitente           fold 4: train=116,468 test=59,600 (111s)


intermitente           fold 5: train=175,878 test=76,056 (146s)
(236158, 29)


## 2. Calibración y pérdida por estrato y alpha

`cobertura` = fracción de filas con `y <= q`; debe acercarse a `alpha`. `mejora_vs_base_rama` = reducción relativa de la pérdida pinball frente a la base por rama;
`mejora_pinball` (con su IC95, bootstrap de productos) = frente a la **base por estrato**, que es la que se usa para decidir.

In [3]:
filas = []
for est in ESTRATOS + ['todos']:
    g = res if est == 'todos' else res[res['estrato'] == est]
    for a in ALPHAS:
        lm = ev.pinball(g['y'], g[f'qm_{a}'], a)
        r = ev.mejora_perdida_con_ic(g, lm, ev.pinball(g['y'], g[f'qb_{a}'], a), B, SEMILLA)
        re = ev.mejora_perdida_con_ic(g, lm, ev.pinball(g['y'], g[f'qe_{a}'], a), B, SEMILLA)
        filas.append(dict(estrato=est, alpha=a, n_filas=len(g), n_prod=r['n_productos'],
                          cob_modelo=float((g['y'] <= g[f'qm_{a}']).mean()), cob_base=float((g['y'] <= g[f'qb_{a}']).mean()),
                          cob_base_estrato=float((g['y'] <= g[f'qe_{a}']).mean()),
                          mejora_vs_base_rama=r['mejora_rel'], mejora_pinball=re['mejora_rel'], lo=re['ic_bajo'], hi=re['ic_alto']))
tab = pd.DataFrame(filas)
for est in ESTRATOS + ['todos']:
    t = tab[tab['estrato'] == est]
    print(f'\n=== {est.upper()} (n={int(t.n_filas.iloc[0]):,}) ===')
    print(t[['alpha', 'cob_modelo', 'cob_base', 'cob_base_estrato', 'mejora_vs_base_rama', 'mejora_pinball', 'lo', 'hi']].to_string(index=False))


=== CABEZA (n=7,793) ===
 alpha  cob_modelo  cob_base  cob_base_estrato  mejora_vs_base_rama  mejora_pinball     lo    hi
 0.100       0.153     0.030             0.107                0.320           0.021 -0.031 0.081
 0.167       0.220     0.047             0.172                0.409           0.040  0.003 0.089
 0.250       0.289     0.075             0.251                0.411           0.033  0.002 0.070
 0.500       0.479     0.277             0.499                0.204           0.012 -0.018 0.037
 0.750       0.672     0.659             0.755                0.020          -0.007 -0.037 0.023
 0.893       0.788     0.895             0.899               -0.002           0.005 -0.076 0.101
 0.950       0.853     0.962             0.946                0.018          -0.027 -0.131 0.073

=== MEDIO (n=47,300) ===
 alpha  cob_modelo  cob_base  cob_base_estrato  mejora_vs_base_rama  mejora_pinball    lo    hi
 0.100       0.126     0.041             0.115                0.275         

## 3. Por rama, con los alphas y costos de negocio

Cada rama se evalúa con su alpha de negocio (0.893 o 0.167) y con el **costo asimétrico real** (`Cu`, `Co`); la mejora del costo es idéntica a la del pinball
en ese alpha (el costo es un múltiplo del pinball), y se reporta también en unidades relativas al volumen.

In [4]:
filas = []
for rama in ALPHA_RAMA:
    a, cc = ALPHA_RAMA[rama], COSTO_RAMA[rama]
    for est in ESTRATOS + ['todos']:
        g = res[res['rama'] == rama]
        g = g if est == 'todos' else g[g['estrato'] == est]
        if len(g) < 30:
            continue
        cm = ev.costo_asimetrico(g['y'], g[f'qm_{a}'], cc['Cu'], cc['Co'])
        cb = ev.costo_asimetrico(g['y'], g[f'qe_{a}'], cc['Cu'], cc['Co'])
        r = ev.mejora_perdida_con_ic(g, cm, cb, B, SEMILLA)
        filas.append(dict(rama=rama, estrato=est, alpha=a, Cu=cc['Cu'], Co=cc['Co'], n_filas=len(g), n_prod=r['n_productos'],
                          cob_modelo=float((g['y'] <= g[f'qm_{a}']).mean()), cob_base=float((g['y'] <= g[f'qe_{a}']).mean()),
                          costo_modelo_por_unidad=r['perdida_modelo'] / g['y'].sum(), costo_base_por_unidad=r['perdida_base'] / g['y'].sum(),
                          mejora_costo=r['mejora_rel'], lo=r['ic_bajo'], hi=r['ic_alto']))
neg = pd.DataFrame(filas)
print(neg.drop(columns=['Cu', 'Co']).to_string(index=False))

               rama estrato  alpha  n_filas  n_prod  cob_modelo  cob_base  costo_modelo_por_unidad  costo_base_por_unidad  mejora_costo     lo     hi
       intermitente  cabeza  0.893     5611     130       0.820     0.896                    0.027                  0.029         0.078  0.018  0.158
       intermitente   medio  0.893    46077     909       0.863     0.894                    0.034                  0.037         0.070  0.062  0.078
       intermitente    cola  0.893   181016    2892       0.892     0.901                    0.056                  0.060         0.063  0.059  0.067
       intermitente   todos  0.893   232704    3050       0.884     0.899                    0.037                  0.040         0.069  0.054  0.092
suave_no_perecedero  cabeza  0.893     1531      41       0.670     0.908                    0.021                  0.015        -0.420 -0.508 -0.286
suave_no_perecedero   medio  0.893      869      33       0.771     0.901                    0.022  

## 4. El cuantil de producción de 08 frente a la base

En las ramas suaves 08 no usa el LightGBM relativo de arriba sino un **ensamble Tweedie + LightGBM con recalibración conformal** (`pred_q_negocio`). Aquí se compara ese
cuantil, con el alpha y los costos de cada rama, contra la base por estrato y contra el LightGBM relativo.

In [5]:
prod = pd.read_parquet(f'{DW}/evaluacion_nivel_1.parquet').dropna(subset=['rama'])[LLAVES + ['rama', 'pred_q_negocio']]
for c in ('codigo_item', 'sucursal'):
    prod[c] = prod[c].astype(str)
pr = res.merge(prod, on=LLAVES + ['rama'], how='inner', validate='one_to_one')
assert len(pr) == len(res), (len(pr), len(res))
filas = []
for rama in ALPHA_RAMA:
    a, cc = ALPHA_RAMA[rama], COSTO_RAMA[rama]
    for est in ESTRATOS + ['todos']:
        g = pr[pr['rama'] == rama]
        g = g if est == 'todos' else g[g['estrato'] == est]
        if len(g) < 30:
            continue
        cp = ev.costo_asimetrico(g['y'], g['pred_q_negocio'], cc['Cu'], cc['Co'])
        cb = ev.costo_asimetrico(g['y'], g[f'qe_{a}'], cc['Cu'], cc['Co'])
        cm = ev.costo_asimetrico(g['y'], g[f'qm_{a}'], cc['Cu'], cc['Co'])
        r1, r2 = ev.mejora_perdida_con_ic(g, cp, cb, B, SEMILLA), ev.mejora_perdida_con_ic(g, cm, cp, B, SEMILLA)
        filas.append(dict(rama=rama, estrato=est, alpha=a, n_filas=len(g), n_prod=r1['n_productos'],
                          cob_produccion=float((g['y'] <= g['pred_q_negocio']).mean()), cob_base_estrato=float((g[f'y'] <= g[f'qe_{a}']).mean()),
                          cob_lgb_relativo=float((g['y'] <= g[f'qm_{a}']).mean()),
                          prod_vs_base=r1['mejora_rel'], prod_lo=r1['ic_bajo'], prod_hi=r1['ic_alto'],
                          lgbrel_vs_prod=r2['mejora_rel'], lgbrel_lo=r2['ic_bajo'], lgbrel_hi=r2['ic_alto']))
prod_tab = pd.DataFrame(filas)
print(prod_tab.to_string(index=False))

               rama estrato  alpha  n_filas  n_prod  cob_produccion  cob_base_estrato  cob_lgb_relativo  prod_vs_base  prod_lo  prod_hi  lgbrel_vs_prod  lgbrel_lo  lgbrel_hi
       intermitente  cabeza  0.893     5611     130           0.816             0.896             0.820        -0.152   -0.328   -0.014           0.200      0.051      0.364
       intermitente   medio  0.893    46077     909           0.856             0.894             0.863         0.062    0.049    0.074           0.008     -0.000      0.018
       intermitente    cola  0.893   181016    2892           0.896             0.901             0.892         0.066    0.061    0.070          -0.003     -0.005     -0.000
       intermitente   todos  0.893   232704    3050           0.886             0.899             0.884         0.006   -0.054    0.048           0.064      0.013      0.135
suave_no_perecedero  cabeza  0.893     1531      41           0.892             0.908             0.670        -0.034   -0.133    

## 5. Sensibilidad a `Cu` y `Co` (ejercicio)

Para cada par (`Cu`, `Co`) el alpha óptimo es `Cu/(Cu+Co)`; se toma el alpha más cercano de la rejilla y se reporta la mejora del modelo sobre la base y su
cobertura, por estrato. Si el alpha óptimo no está en la rejilla la lectura es aproximada (columna `dif_alpha`).

In [6]:
grid_cu, grid_co = [0.1, 0.2, 0.25, 0.4], [0.03, 0.1, 0.25, 0.5, 1.0]
filas = []
for Cu in grid_cu:
    for Co in grid_co:
        a_opt = Cu / (Cu + Co)
        a = min(ALPHAS, key=lambda x: abs(x - a_opt))
        for est in ESTRATOS + ['todos']:
            t = tab[(tab['estrato'] == est) & (tab['alpha'] == a)].iloc[0]
            filas.append(dict(Cu=Cu, Co=Co, alpha_optimo=round(a_opt, 3), alpha_rejilla=a, dif_alpha=round(abs(a - a_opt), 3), estrato=est,
                              mejora=t['mejora_pinball'], lo=t['lo'], hi=t['hi'], cob_modelo=t['cob_modelo']))
sens = pd.DataFrame(filas)
piv = sens[sens.estrato == 'todos'].pivot(index='Cu', columns='Co', values='mejora')
print('Mejora del modelo sobre la base (todos los estratos), por (Cu, Co):'); print(piv.round(3))
piv2 = sens[sens.estrato == 'cabeza'].pivot(index='Cu', columns='Co', values='mejora')
print('\nSolo cabeza:'); print(piv2.round(3))
print('\nalpha óptimo por (Cu, Co):'); print(sens[sens.estrato == 'todos'].pivot(index='Cu', columns='Co', values='alpha_optimo').round(3))

Mejora del modelo sobre la base (todos los estratos), por (Cu, Co):
Co     0.030  0.100  0.250  0.500  1.000
Cu                                      
0.100  0.029  0.045  0.092  0.076  0.060
0.200  0.045  0.029  0.045  0.092  0.076
0.250  0.045  0.029  0.045  0.092  0.076
0.400  0.057  0.029  0.045  0.045  0.092

Solo cabeza:
Co     0.030  0.100  0.250  0.500  1.000
Cu                                      
0.100 -0.007  0.012  0.033  0.040  0.021
0.200  0.005 -0.007  0.012  0.033  0.040
0.250  0.005 -0.007  0.012  0.033  0.040
0.400 -0.027 -0.007  0.012  0.012  0.033

alpha óptimo por (Cu, Co):
Co     0.030  0.100  0.250  0.500  1.000
Cu                                      
0.100  0.769  0.500  0.286  0.167  0.091
0.200  0.870  0.667  0.444  0.286  0.167
0.250  0.893  0.714  0.500  0.333  0.200
0.400  0.930  0.800  0.615  0.444  0.286


## 6. Decisión mecánica, artefactos y registro en MLflow

In [7]:
dec = {'notebook': '13_cuantiles_y_costos', 'fecha_generacion': pd.Timestamp.now().isoformat(), 'alphas': ALPHAS,
       'costos': {k: {'Cu': v['Cu'], 'Co': v['Co']} for k, v in COSTO_RAMA.items()}, 'modelo': 'LightGBM cuantilico sobre y/base, features v1, hparams por rama',
       'base': '(media_movil+1) * Q_alpha(y/(media_movil+1)) sobre train purgado', 'politica_holdout': ev.POLITICA_HOLDOUT, 'negocio': {}}
for _, r in neg.iterrows():
    dec['negocio'][f"{r['rama']}|{r['estrato']}"] = dict(alpha=r['alpha'], n_filas=int(r['n_filas']), cobertura_modelo=r['cob_modelo'], cobertura_base=r['cob_base'],
                                                        mejora_costo=r['mejora_costo'], ic=[r['lo'], r['hi']], modelo_mejor_que_base_con_ic=bool(r['lo'] > 0),
                                                        desvio_cobertura_modelo=r['cob_modelo'] - r['alpha'], desvio_cobertura_base=r['cob_base'] - r['alpha'])
dec['produccion_vs_base_estrato'] = {f"{r['rama']}|{r['estrato']}": dict(alpha=r['alpha'], cobertura_produccion=r['cob_produccion'], cobertura_base_estrato=r['cob_base_estrato'],
                                     mejora_costo=r['prod_vs_base'], ic=[r['prod_lo'], r['prod_hi']], produccion_mejor_que_base_con_ic=bool(r['prod_lo'] > 0))
                                     for _, r in prod_tab.iterrows()}
prod_tab.to_csv(f'{DW}/cuantiles_produccion_vs_base_13.csv', index=False)
with open(f'{DW}/decision_13.json', 'w') as f:
    json.dump(dec, f, indent=2, ensure_ascii=False)
tab.to_csv(f'{DW}/cuantiles_por_estrato_13.csv', index=False)
neg.to_csv(f'{DW}/cuantiles_negocio_por_rama_13.csv', index=False)
sens.to_csv(f'{DW}/sensibilidad_costos_13.csv', index=False)
res.to_parquet(f'{DW}/preds_cuantiles_13.parquet', index=False)
huella = {'notebook': '13_cuantiles_y_costos', 'fecha_generacion': pd.Timestamp.now().isoformat(),
          'entradas': registrar_huella(['matriz_as_of.parquet', 'estratos_por_fold_12.parquet']),
          'salidas': registrar_huella(['decision_13.json', 'cuantiles_por_estrato_13.csv', 'preds_cuantiles_13.parquet'])}
with open(f'{DW}/huella_13.json', 'w') as f:
    json.dump(huella, f, indent=2, ensure_ascii=False)
for _, r in tab.iterrows():
    ev.registrar_mlflow(EXPERIMENTO, f"{r['estrato']}__alpha_{r['alpha']}",
        tags={'fase': 'E', 'notebook': '13', 'estrato': r['estrato'], 'alpha': r['alpha'], 'rama_git': 'fix/eda-modelado-nivel1'},
        params={'alpha': r['alpha'], 'modelo': 'lgb cuantilico relativo (v1)', 'base': 'media movil x cuantil de razon', 'bootstrap_B': B},
        metrics={k: r[k] for k in ['n_filas', 'n_prod', 'cob_modelo', 'cob_base', 'mejora_pinball', 'lo', 'hi']},
        artefactos=[f'{DW}/decision_13.json'] if (r['estrato'] == 'todos' and r['alpha'] == 0.893) else ())
print(f'{len(tab)} runs registrados en {EXPERIMENTO}')

🏃 View run cabeza__alpha_0.1 at: http://127.0.0.1:5000/#/experiments/20/runs/767dadc440bf40d0b7e9cf0b221a4da9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/20


🏃 View run cabeza__alpha_0.167 at: http://127.0.0.1:5000/#/experiments/20/runs/aa64b5bdd7d748818469fd3b2319b381
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/20


🏃 View run cabeza__alpha_0.25 at: http://127.0.0.1:5000/#/experiments/20/runs/734af472cd7444fb909c1d7804af124f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/20


🏃 View run cabeza__alpha_0.5 at: http://127.0.0.1:5000/#/experiments/20/runs/f0e084ee400d44ddb7b55acd5d568127
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/20


🏃 View run cabeza__alpha_0.75 at: http://127.0.0.1:5000/#/experiments/20/runs/87c3b307c48d4baa89b015e83b274f32
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/20


...

🏃 View run todos__alpha_0.25 at: http://127.0.0.1:5000/#/experiments/20/runs/b001b550d5f241b187f87b5868b4774d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/20


🏃 View run todos__alpha_0.5 at: http://127.0.0.1:5000/#/experiments/20/runs/29aaebf418ad4ae69389acd2d10aa86d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/20


🏃 View run todos__alpha_0.75 at: http://127.0.0.1:5000/#/experiments/20/runs/9e3f3751ffea4fb7b595cb113517399d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/20


🏃 View run todos__alpha_0.893 at: http://127.0.0.1:5000/#/experiments/20/runs/df6156c29fa64e598c2454adf1cd190e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/20


🏃 View run todos__alpha_0.95 at: http://127.0.0.1:5000/#/experiments/20/runs/393e61c97f574ac78e484c6679997a25
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/20
28 runs registrados en EDA-fix-nivel1-13-cuantiles-y-costos
